# Lab 19: GraphRAG — Notebook Orchestrator

This notebook is a **thin orchestration layer** that calls the `src/` modules and visualises results.
All heavy logic lives in separate Python files — run individual steps from the terminal:

```bash
python scripts/01_fetch.py          # fetch 10 Wikipedia articles -> data/raw/
python scripts/02_extract.py        # LLM NER -> data/processed/triples.json
python scripts/03_build_graph.py    # Neo4j + embeddings
python scripts/04_test_query.py     # compare GraphRAG vs Flat RAG interactively
python scripts/05_benchmark.py      # 20-question benchmark -> benchmark_results.csv
python visualizations/visualize_graph.py [entity] [hops]   # PNG output
```

**Neo4j Browser**: http://localhost:7474  
*(start Neo4j first: `docker run -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password neo4j:latest`)*

In [ ]:
# Install packages (once)
import subprocess, sys
pkgs = ["openai>=1.30", "neo4j>=5.19", "chromadb>=0.5", "wikipedia",
        "python-dotenv>=1.0", "tiktoken>=0.7", "pandas>=2.2", "numpy", "networkx>=3.3", "matplotlib"]
for p in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", p, "-q"], capture_output=True)
print("All packages ready")

In [ ]:
import sys, json, functools
from pathlib import Path
sys.path.insert(0, str(Path().resolve()))   # ensure project root is on path

from src.config import DATA_RAW, DATA_PROCESSED, VISUALIZATIONS, COMPANIES
print("src/ modules loaded")
print(f"  data/raw       : {DATA_RAW}")
print(f"  data/processed : {DATA_PROCESSED}")
print(f"  visualizations : {VISUALIZATIONS}")

## Step 1 — Data Collection

In [ ]:
from src.fetch_corpus import fetch_all, chunk_corpus, load_corpus, load_chunks

# Fetch only if not already cached
raw_files = list(DATA_RAW.glob("*.json"))
if len(raw_files) >= len(COMPANIES):
    print(f"Loading cached corpus ({len(raw_files)} files from data/raw/)")
    corpus = load_corpus()
else:
    print("Fetching Wikipedia articles...")
    corpus = fetch_all()

chunks_file = DATA_PROCESSED / "chunks.json"
if chunks_file.exists():
    chunks = load_chunks()
else:
    chunks = chunk_corpus(corpus)

In [ ]:
# EDA: corpus overview
import pandas as pd

rows = []
for doc in corpus:
    n_chunks = len([c for c in chunks if c["company"] == doc["company"]])
    rows.append({"company": doc["company"], "title": doc["title"],
                 "chars": len(doc["text"]), "chunks": n_chunks})

df_corpus = pd.DataFrame(rows)
print(f"Total: {len(corpus)} articles, {len(chunks)} chunks")
display(df_corpus)

## Step 2 — Triple Extraction

In [ ]:
from src.extract_triples import extract_all, load_triples, load_stats

triples_file = DATA_PROCESSED / "triples.json"
if triples_file.exists():
    print("Loading cached triples...")
    triples = load_triples()
    stats   = load_stats()
else:
    print("Extracting triples (calls OpenAI)...")
    triples, stats = extract_all(chunks)

In [ ]:
# EDA: triples overview
df_triples = pd.DataFrame(triples)
unique_entities = set(df_triples["subject"].tolist() + df_triples["object"].tolist())

print(f"Triples   : {len(df_triples)}")
print(f"Entities  : {len(unique_entities)}")
print(f"Tokens    : {stats['total_tokens']:,}")
print(f"Est. cost : ${stats['total_tokens'] * 0.15 / 1_000_000:.4f}\n")

print("Top predicates:")
print(df_triples["predicate"].value_counts().head(12).to_string())

print("\nSample triples:")
display(df_triples.sample(min(15, len(df_triples)), random_state=42)[["subject", "predicate", "object", "source"]])

## Step 3 — Build Knowledge Graph

In [ ]:
from src.build_graph import build_full_graph, load_node_embeddings, get_graph_stats

emb_file = DATA_PROCESSED / "node_embeddings.json"
if emb_file.exists():
    print("Loading cached node embeddings...")
    node_embeddings = load_node_embeddings()
    print(f"  {len(node_embeddings)} nodes loaded")
else:
    print("Building Neo4j graph + embeddings (calls OpenAI + Neo4j)...")
    node_embeddings = build_full_graph(triples)

In [ ]:
# Graph statistics
stats_graph = get_graph_stats()
print(f"Nodes : {stats_graph['nodes']}")
print(f"Edges : {stats_graph['edges']}")
print("\nNodes by company:")
for row in stats_graph["by_company"]:
    print(f"  {row['company']}: {row['nodes']} nodes")

print("\nNeo4j Browser: http://localhost:7474")
print("Query: MATCH (n:Entity)-[r]->(m) RETURN n,r,m LIMIT 100")

## Step 4 — GraphRAG vs Flat RAG: Interactive Comparison

In [ ]:
import src.graphrag as graphrag_mod
import src.flatrag  as flatrag_mod

print("Building Flat RAG index (ChromaDB, ~1-2 min)...")
flat_col = flatrag_mod.build_index(chunks)

gr_fn = functools.partial(graphrag_mod.query, node_embeddings=node_embeddings)
fr_fn = functools.partial(flatrag_mod.query,  collection=flat_col)

In [ ]:
# Test with a multi-hop question
question = "Which AI companies were co-founded by former Google employees?"

gr = gr_fn(question)
fr = fr_fn(question)

print(f"Q: {question}\n")
print(f"[GraphRAG]  seeds={gr['seeds']}  subgraph={gr['subgraph_size']} triples  {gr['latency']}s")
print(f"  {gr['answer']}\n")
print(f"[Flat RAG]  {fr['latency']}s")
print(f"  {fr['answer']}")

## Step 5 — Benchmark (20 Questions)

In [ ]:
from src.benchmark import run, print_summary

# This takes ~5-10 minutes
df_bench = run(gr_fn, fr_fn)
print_summary(df_bench)

In [ ]:
# Coloured results table
def _color(val):
    if val is True:  return "background-color: #c8f5c8"
    if val is False: return "background-color: #f5c8c8"
    return ""

display(
    df_bench[["id", "type", "graphrag_correct", "flatrag_correct",
              "graphrag_latency", "flatrag_latency",
              "graphrag_tokens",  "flatrag_tokens"]]
    .style
    .applymap(_color, subset=["graphrag_correct", "flatrag_correct"])
    .format({"graphrag_latency": "{:.2f}s", "flatrag_latency": "{:.2f}s"})
)

## Visualization

In [ ]:
# Run the visualizer and display the output PNGs inline
from visualizations.visualize_graph import visualize_full, visualize_subgraph, print_neo4j_queries
from IPython.display import Image, display as ipy_display

triples = load_triples()

out_full = visualize_full(triples, max_nodes=80)
ipy_display(Image(filename=str(out_full)))

In [ ]:
# Subgraph for a specific entity (change the name as needed)
ENTITY = "OpenAI"    # try: Anthropic (company), Google DeepMind, Sam Altman, ...
HOPS   = 2

out_sub = visualize_subgraph(ENTITY, triples, hops=HOPS)
if out_sub:
    ipy_display(Image(filename=str(out_sub)))

In [ ]:
# Print Neo4j Browser Cypher queries for interactive exploration
print_neo4j_queries()